# Avaliação de IoU — Versão 2 (atual)


Código atual de avaliação: pareamento guloso entre predições e ground truth por IoU
(limiar 0,5), reportando TP/FP/FN separadamente para bounding box e máscara de
segmentação, com verificação de consistência entre os totais de GT calculados pelos
dois métodos.


In [ ]:
# ============================================================
# CÉLULA 1 — Imports e funções de conversão de anotação
# ============================================================
import numpy as np
import cv2
from pathlib import Path
from ultralytics import YOLO


def parse_yolo_segmentation_label(label_path):
    """
    Lê um .txt no formato YOLO-segmentação:
        class x1 y1 x2 y2 x3 y3 ... xn yn  (normalizado em [0,1])
    Retorna lista de polígonos (cada um é um array de pontos).
    Importante: isso NÃO é o formato de detecção (5 valores fixos) —
    é um polígono com número variável de pontos.
    """
    polygons = []
    if not Path(label_path).exists():
        return polygons

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:  # classe + pelo menos 3 pontos
                continue
            coords = list(map(float, parts[1:]))
            poly = np.array(coords, dtype=np.float32).reshape(-1, 2)
            polygons.append(poly)
    return polygons


def polygon_to_bbox(polygon_norm, img_w, img_h):
    """Caixa envolvente (min/max) de um polígono normalizado."""
    poly_px = polygon_norm.copy()
    poly_px[:, 0] *= img_w
    poly_px[:, 1] *= img_h
    x1, y1 = poly_px[:, 0].min(), poly_px[:, 1].min()
    x2, y2 = poly_px[:, 0].max(), poly_px[:, 1].max()
    return np.array([x1, y1, x2, y2])


def polygon_to_mask(polygon_norm, img_w, img_h):
    """Rasteriza um polígono normalizado em máscara binária."""
    poly_px = polygon_norm.copy()
    poly_px[:, 0] *= img_w
    poly_px[:, 1] *= img_h
    poly_px = poly_px.astype(np.int32)

    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    cv2.fillPoly(mask, [poly_px], 1)
    return mask


def compute_iou_bbox(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def compute_iou_mask(mask_a, mask_b):
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return inter / union if union > 0 else 0.0


# ============================================================
# CÉLULA 2 — Pareamento guloso (com diagnóstico dos FN)
# ============================================================
def greedy_match(iou_matrix, iou_threshold=0.5):
    n_pred, n_gt = iou_matrix.shape
    if n_gt == 0:
        return [], n_pred, 0, []
    if n_pred == 0:
        return [], 0, n_gt, [0.0] * n_gt

    pairs = [(i, j, iou_matrix[i, j]) for i in range(n_pred) for j in range(n_gt)]
    pairs.sort(key=lambda x: x[2], reverse=True)

    used_p, used_g, matched = set(), set(), []
    for i, j, val in pairs:
        if i in used_p or j in used_g:
            continue
        if val < iou_threshold:
            break
        matched.append(val)
        used_p.add(i)
        used_g.add(j)

    fp = n_pred - len(used_p)
    fn = n_gt - len(used_g)

    # Para os GTs que ficaram como FN: qual foi o melhor IoU que tiveram
    # com alguma predição, mesmo abaixo do threshold (só diagnóstico).
    best_unmatched_gt_iou = []
    for j in range(n_gt):
        if j in used_g:
            continue
        best = iou_matrix[:, j].max() if n_pred > 0 else 0.0
        best_unmatched_gt_iou.append(best)

    return matched, fp, fn, best_unmatched_gt_iou


# ============================================================
# CÉLULA 3 — Predição no dataset v3
# ============================================================
# AJUSTE aqui: caminho do seu checkpoint treinado no v3
model = YOLO("/content/drive/MyDrive/LuisPedro/pistaClandestina/datasets/google_earth_v3_brightness/treino_yolov8_seg-n_v3_brightness_8batch/weights/best.pt")

images_dir = Path("/content/drive/MyDrive/LuisPedro/pistaClandestina/datasets/google_earth_v3_brightness/test/images")
labels_dir = images_dir.parent / "labels"

results = model.predict(source=str(images_dir), conf=0.25, save=True)


# ============================================================
# CÉLULA 4 — Avaliação (separando "todas" de "só com pista real")
# ============================================================
bbox_ious, mask_ious = [], []
bbox_ious_com_pista, mask_ious_com_pista = [], []  # só onde há GT de verdade
fp_bbox_total = fn_bbox_total = 0
fp_mask_total = fn_mask_total = 0

for r in results:
    img_h, img_w = r.orig_shape
    img_name = Path(r.path).stem
    label_path = labels_dir / f"{img_name}.txt"

    gt_polygons = parse_yolo_segmentation_label(label_path)
    gt_bboxes = [polygon_to_bbox(p, img_w, img_h) for p in gt_polygons]
    gt_masks = [polygon_to_mask(p, img_w, img_h) for p in gt_polygons]  # SEMPRE calculado
    tem_pista_real = len(gt_polygons) > 0

    pred_bboxes = r.boxes.xyxy.cpu().numpy() if r.boxes is not None else np.empty((0, 4))
    if r.masks is not None:
        pred_masks_raw = r.masks.data.cpu().numpy()
        pred_masks = [cv2.resize(m, (img_w, img_h), interpolation=cv2.INTER_NEAREST).astype(np.uint8)
                      for m in pred_masks_raw]
    else:
        pred_masks = []

    # --- Bounding box ---
    iou_mat_bbox = np.array([[compute_iou_bbox(pb, gb) for gb in gt_bboxes] for pb in pred_bboxes]) \
                   if len(pred_bboxes) and len(gt_bboxes) else np.zeros((len(pred_bboxes), len(gt_bboxes)))
    matched, fp, fn, _ = greedy_match(iou_mat_bbox, iou_threshold=0.5)
    bbox_ious.extend(matched)
    fp_bbox_total += fp
    fn_bbox_total += fn
    if tem_pista_real:
        bbox_ious_com_pista.extend(matched)

    # --- Máscara ---
    iou_mat_mask = np.array([[compute_iou_mask(pm, gm) for gm in gt_masks] for pm in pred_masks]) \
                   if len(pred_masks) and len(gt_masks) else np.zeros((len(pred_masks), len(gt_masks)))
    matched_m, fp_m, fn_m, _ = greedy_match(iou_mat_mask, iou_threshold=0.5)
    mask_ious.extend(matched_m)
    fp_mask_total += fp_m
    fn_mask_total += fn_m
    if tem_pista_real:
        mask_ious_com_pista.extend(matched_m)


# ============================================================
# CÉLULA 5 — Verificação de consistência (não pula essa etapa!)
# ============================================================
total_gt_bbox = len(bbox_ious) + fn_bbox_total
total_gt_mask = len(mask_ious) + fn_mask_total
print(f"Total de GT (bbox): {total_gt_bbox} | Total de GT (máscara): {total_gt_mask}")
assert total_gt_bbox == total_gt_mask, "ERRO: os totais não batem — não confie nos números abaixo."
print("OK: os totais batem.\n")


# ============================================================
# CÉLULA 6 — Relatório final
# ============================================================
def report(name, ious_todas, ious_com_pista, fp, fn):
    ious_todas = np.array(ious_todas)
    ious_com_pista = np.array(ious_com_pista)
    print(f"--- {name} ---")
    print(f"TP: {len(ious_todas)} | FP: {fp} | FN: {fn}")
    if len(ious_todas) > 0:
        print(f"IoU médio (todas as imagens): {ious_todas.mean():.4f}")
    if len(ious_com_pista) > 0:
        print(f"IoU médio (só imagens com pista real): {ious_com_pista.mean():.4f}")
    print()

report("Bounding Box", bbox_ious, bbox_ious_com_pista, fp_bbox_total, fn_bbox_total)
report("Máscara (pixel a pixel)", mask_ious, mask_ious_com_pista, fp_mask_total, fn_mask_total)